In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# 警告を無視
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
# パス設定
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_wavenet.csv'

# ハイパーパラメータ
MAX_LEN = 40
BATCH_SIZE = 64
N_FOLDS = 10
EPOCHS = 30           
LEARNING_RATE = 1e-3  
WEIGHT_DECAY = 1e-5   

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

# シード固定
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 特徴量エンジニアリング (PID Features)
# ==============================================================================
def add_pid_features(df):
    """
    Google Brain Ventilatorコンペの上位解法を模倣。
    時系列の「微分(Diff)」「積分(Cumsum)」「ラグ(Lag)」を追加します。
    """
    df_eng = df.copy()
    
    # 除外カラム
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    print(f"Generating PID features for {len(base_cols)} base columns...")
    
    # Groupbyオブジェクトを先に作成（高速化）
    grouped = df_eng.groupby('sample_id')
    
    new_cols = []
    
    for col in base_cols:
        # 1. 時間重み付け (Time-Weighting: あなたのアイデア)
        # 後半のデータを重視する「学習効果」ゲイン
        max_day = df_eng['day_n'].max()
        gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
        
        # 基本特徴量 (Weighted)
        w_col_name = f'{col}_weighted'
        df_eng[w_col_name] = df_eng[col] * gain
        # PID生成のベースはこの重み付きデータを使います
        
        # --- D (微分): 変化の勢い ---
        # 1つ前の時刻との差 (速度)
        df_eng[f'{col}_diff1'] = grouped[col].diff().fillna(0) * gain
        # 2つ前の時刻との差 (加速度的な要素)
        df_eng[f'{col}_diff2'] = grouped[col].diff(2).fillna(0) * gain
        
        # --- I (積分): 蓄積 ---
        # そのサンプルの開始時点からの累積和 (疲労や活性化の蓄積)
        # ※値が大きくなりすぎるのを防ぐため、スケーリング前に平均で割る等の処理も有効ですが
        #   今回はRobustScalerに任せます。
        df_eng[f'{col}_cumsum'] = grouped[col].cumsum()
        
        # --- Lag (過去の状態) ---
        # Ventilatorコンペでは「R」「C」という固定値との相互作用が効きましたが、
        # ここでは「直前の値」を明示的に入れます。
        df_eng[f'{col}_lag1'] = grouped[col].shift(1).fillna(0)
        
    return df_eng

print(">>> Loading Data & Engineering Features...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

# 特徴量生成 (少し時間がかかります)
train = add_pid_features(train)
test = add_pid_features(test)

# 特徴量カラムの選定
target_col = 'lever'
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
# 生成された全カラムを取得
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
# Train/Test共通のカラムのみ使用
feature_cols = [c for c in train_feats if c in test_feats]

print(f"Total Input Features: {len(feature_cols)}")

# スケーリング
# 累積和(cumsum)などは外れ値が出やすいので、StandardScalerよりRobustScalerが安全です
scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])


# ==============================================================================
# 3. Dataset Class
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        # Padding
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        
        # Mask
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {
            'x': torch.tensor(x_pad),
            'mask': torch.tensor(mask),
            'id': sample_id
        }
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
            
        return ret

# ==============================================================================
# 4. Model Definition: WaveNet (Dilated CNN)
# ==============================================================================
class WaveBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        self.dilation = dilation
        self.padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Conv1d(
            channels, channels, 
            kernel_size=kernel_size, 
            padding=self.padding, 
            dilation=dilation
        )
        self.bn = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + residual

class WaveNetModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, kernel_size=3, depth=12):
        super().__init__()
        self.start_conv = nn.Conv1d(input_dim, hidden_dim, kernel_size=1)
        self.blocks = nn.ModuleList()
        for i in range(depth):
            dilation = 2 ** (i % 4)
            self.blocks.append(WaveBlock(hidden_dim, kernel_size, dilation))
        self.end_conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Conv1d(hidden_dim, 1, kernel_size=1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.start_conv(x)
        for block in self.blocks:
            x = block(x)
        x = self.end_conv(x)
        return x.squeeze(1)

# ==============================================================================
# 5. Training Loop (Sequential Saving Added)
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

# Test Dataset (一度作れば使い回せる)
test_ds = BrainDataset(test, brain_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n>>> Starting {N_FOLDS}-Fold Cross Validation (WaveNet)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    # データ準備
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], brain_cols, target_col, MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], brain_cols, target_col, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル初期化
    model = WaveNetModel(
        input_dim=len(brain_cols),
        hidden_dim=128,
        kernel_size=3,
        depth=12 
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_wavenet_fold{fold}.pth"
    
    # --- 学習ループ ---
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        
        print(f"  Ep {epoch+1} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
    
    print(f"  >> Best Val Loss: {best_loss:.5f}")
    
    # --- 推論 & 中間保存 ---
    print("  Predicting Test Data...")
    
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                preds = model(x).cpu().numpy()
                mask = mask.cpu().numpy()
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    for t, val in enumerate(preds[i, :v_len]):
                        k = f"{s_id}_{t}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
        
        # ▼【追加】Foldごとの中間保存 ▼
        print(f"  Saving intermediate results for Fold {fold+1}...")
        temp_results = []
        for k, v in final_test_preds.items():
            # 現時点での平均値を計算して保存
            temp_results.append({'id': k, 'lever': max(0, v / test_counts[k])})
        
        temp_df = pd.DataFrame(temp_results)
        temp_filename = f"temp_submission_fold{fold+1}.csv"
        temp_df.to_csv(temp_filename, index=False)
        print(f"  -> Saved: {temp_filename}")
        
    except Exception as e:
        print(f"  ERROR in Inference/Saving at Fold {fold+1}: {e}")
        print("  Skipping this fold's inference to continue process.")

    # メモリ解放
    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# 6. Final Submission & Ensemble
# ==============================================================================
print("\n>>> Creating Final WaveNet Submission...")

if len(final_test_preds) > 0:
    results = []
    for k, v in final_test_preds.items():
        results.append({'id': k, 'lever': max(0, v / test_counts[k])})

    # WaveNet単体の結果
    df_wave = pd.DataFrame(results)
    df_wave.to_csv(SUBMISSION_PATH, index=False)
    print(f"WaveNet Single Score Saved: {SUBMISSION_PATH}")
else:
    print("Error: No predictions generated.")

# --- 最終アンサンブル ---
print("\n>>> Generating Final Ensemble (Blending 3 Models)...")

try:
    if os.path.exists('submission3.csv') and os.path.exists('submission_spectrogram_2dcnn.csv'):
        # 1. CNN-LSTM (0.897)
        df_best = pd.read_csv('submission3.csv')
        
        # 2. Spectrogram (0.888〜1.0)
        df_spec = pd.read_csv('submission_spectrogram_2dcnn.csv')
        
        # 3. WaveNet (今回)
        # df_wave は上で作成済み
        
        submission = pd.read_csv(TEST_PATH)[['id']]
        submission = submission.merge(df_best[['id', 'lever']], on='id', suffixes=('', '_best'))
        submission = submission.merge(df_spec[['id', 'lever']], on='id', suffixes=('', '_spec'))
        submission = submission.merge(df_wave[['id', 'lever']], on='id', suffixes=('', '_wave'))
        
        submission['final_lever'] = (
            submission['lever'] * 0.50 + 
            submission['lever_spec'] * 0.20 + 
            submission['lever_wave'] * 0.30
        )
        
        final_sub = submission[['id', 'final_lever']].rename(columns={'final_lever': 'lever'})
        final_sub['lever'] = final_sub['lever'].fillna(0.0)
        final_sub.to_csv("submission_ensemble_3models.csv", index=False)
        
        print("!!! FINAL ENSEMBLE COMPLETED !!!")
        print("Saved to: submission_ensemble_3models.csv")
    else:
        print("Skipping ensemble: Dependent files not found.")
        
except Exception as e:
    print(f"Ensemble failed: {e}")

Using Device: cuda
>>> Loading Data & Engineering Features...
Generating PID features for 44 base columns...
Generating PID features for 44 base columns...
Total Input Features: 264


NameError: name 'RobustScaler' is not defined

In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold
import math

# 警告を無視
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
# パス設定
INPUT_DIR = '.'
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_wavenet.csv'

# ハイパーパラメータ (過学習対策強化版)
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2  # 修正: 1e-5 -> 1e-2 (強力な正則化)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

# シード固定
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 特徴量エンジニアリング (PID Features)
# ==============================================================================
def add_pid_features(df):
    """
    PID特徴量 + 時間重み付け
    """
    df_eng = df.copy()
    
    # 除外カラム
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    print(f"Generating PID features for {len(base_cols)} base columns...")
    
    # 時間重み付け (Gain)
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    # Groupbyオブジェクト作成
    grouped = df_eng.groupby('sample_id')
    
    for col in base_cols:
        # P (Proportional)
        df_eng[f'{col}_weighted'] = df_eng[col] * gain
        
        # D (Derivative)
        col_diff1 = grouped[col].diff().fillna(0)
        col_diff2 = grouped[col].diff(2).fillna(0)
        
        df_eng[f'{col}_diff1'] = col_diff1 * gain
        df_eng[f'{col}_diff2'] = col_diff2 * gain
        
        # I (Integral)
        df_eng[f'{col}_cumsum'] = grouped[col].cumsum()
        
        # Lag
        df_eng[f'{col}_lag1'] = grouped[col].shift(1).fillna(0)
        
    return df_eng

print(">>> Loading Data & Engineering Features...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

# 特徴量生成
train = add_pid_features(train)
test = add_pid_features(test)

target_col = 'lever'
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
# Train/Test共通のカラムのみ使用
feature_cols = [c for c in train_feats if c in test_feats]

print(f"Total Input Features: {len(feature_cols)}")

# スケーリング (RobustScaler推奨)
scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dynamic MAX_LEN & Dataset
# ==============================================================================
# データの最大長を動的に取得
max_seq_len_train = train.groupby('sample_id').size().max()
max_seq_len_test = test.groupby('sample_id').size().max()
MAX_RAW_LEN = max(max_seq_len_train, max_seq_len_test)

# WaveNet用に余裕を持たせる（32の倍数）
MAX_LEN = math.ceil(MAX_RAW_LEN / 32) * 32
print(f"Dynamic MAX_LEN set to: {MAX_LEN} (Raw max: {MAX_RAW_LEN})")

class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=MAX_LEN):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        # Padding
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        
        # Mask
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {
            'x': torch.tensor(x_pad),
            'mask': torch.tensor(mask),
            'id': sample_id,
            'seq_len': seq_len  # 修正: seq_lenを返す
        }
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
            
        return ret

# ==============================================================================
# 4. Model Definition: WaveNet (Dilated CNN)
# ==============================================================================
class WaveBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        self.dilation = dilation
        self.padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Conv1d(
            channels, channels, 
            kernel_size=kernel_size, 
            padding=self.padding, 
            dilation=dilation
        )
        self.bn = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        
        # 修正: Dropout率を大幅アップ (0.1 -> 0.4)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        residual = x
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + residual

class WaveNetModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, kernel_size=3, depth=12):
        super().__init__()
        self.start_conv = nn.Conv1d(input_dim, hidden_dim, kernel_size=1)
        
        self.blocks = nn.ModuleList()
        for i in range(depth):
            dilation = 2 ** (i % 4) 
            self.blocks.append(WaveBlock(hidden_dim, kernel_size, dilation))
            
        self.end_conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1),
            nn.ReLU(),
            nn.Dropout(0.4), # ここも強化
            nn.Conv1d(hidden_dim, 1, kernel_size=1)
        )

    def forward(self, x):
        # x: [Batch, Time, Features] -> [Batch, Features, Time]
        x = x.permute(0, 2, 1)
        
        x = self.start_conv(x)
        
        for block in self.blocks:
            x = block(x)
            
        x = self.end_conv(x)
        
        # [Batch, 1, Time] -> [Batch, Time]
        return x.squeeze(1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
# IDマッピング（提出用）
test_sample_map = test.groupby('sample_id')['id'].apply(list).to_dict()
submission_dict = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold Cross Validation (WaveNet)...")
print(f"    Max Sequence Length: {MAX_LEN}")

# Test Dataset
test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    # データ準備
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, target_col, MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, target_col, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル初期化
    model = WaveNetModel(
        input_dim=len(feature_cols),
        hidden_dim=128,
        kernel_size=3,
        depth=12 
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_wavenet_fold{fold}.pth"
    
    # --- 学習ループ ---
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x = batch['x'].to(DEVICE)
            y = batch['y'].to(DEVICE)
            m = batch['mask'].to(DEVICE)
            
            optimizer.zero_grad()
            preds = model(x)
            
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(DEVICE)
                y = batch['y'].to(DEVICE)
                m = batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
    
    print(f"  >> Best Val Loss: {best_loss:.5f}")
    
    # --- 推論 ---
    print("  Predicting Test Data...")
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                s_ids = batch['id']
                seq_lens = batch['seq_len'].numpy()
                
                preds = model(x).cpu().numpy()
                
                for i, s_id in enumerate(s_ids):
                    # 有効な長さ分だけ取り出す
                    valid_len = seq_lens[i]
                    pred_seq = np.maximum(0, preds[i, :valid_len])
                    
                    # 正しいrow_idを取得
                    row_ids = test_sample_map[s_id]
                    
                    # 長さの不一致対策
                    if len(row_ids) != len(pred_seq):
                        min_len = min(len(row_ids), len(pred_seq))
                        row_ids = row_ids[:min_len]
                        pred_seq = pred_seq[:min_len]
                    
                    # 辞書に加算
                    for r_id, p_val in zip(row_ids, pred_seq):
                        submission_dict[r_id] = submission_dict.get(r_id, 0) + p_val
        
        print(f"  Fold {fold+1} prediction finished.")
        
    except Exception as e:
        print(f"  ERROR in Inference Fold {fold+1}: {e}")

    # メモリ解放
    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# 6. Final Submission
# ==============================================================================
print("\n>>> Creating Final WaveNet Submission...")

if len(submission_dict) > 0:
    results = []
    # test.csvの並び順で作成
    for r_id in test['id']:
        val = submission_dict.get(r_id, 0) / N_FOLDS
        results.append({'id': r_id, 'lever': val})

    df_wave = pd.DataFrame(results)
    df_wave.to_csv(SUBMISSION_PATH, index=False)
    print(f"SUCCESS: {SUBMISSION_PATH} created.")
else:
    print("Error: No predictions generated.")

Using Device: cuda
>>> Loading Data & Engineering Features...
Generating PID features for 44 base columns...
Generating PID features for 44 base columns...
Total Input Features: 264
Dynamic MAX_LEN set to: 64 (Raw max: 40)

>>> Starting 5-Fold Cross Validation (WaveNet)...
    Max Sequence Length: 64

=== Fold 1/5 ===
  Ep  1 | Train: 1.5570 | Val: 1.2511 | LR: 0.000997
  Ep  5 | Train: 0.9280 | Val: 0.9655 | LR: 0.000933
  Ep 10 | Train: 0.8414 | Val: 1.0066 | LR: 0.000750
  Ep 15 | Train: 0.7403 | Val: 0.9277 | LR: 0.000501
  Ep 20 | Train: 0.6562 | Val: 0.8925 | LR: 0.000251
  Ep 25 | Train: 0.5723 | Val: 0.9190 | LR: 0.000068
  Ep 30 | Train: 0.5421 | Val: 0.9379 | LR: 0.000001
  >> Best Val Loss: 0.87750
  Predicting Test Data...
  Fold 1 prediction finished.

=== Fold 2/5 ===
  Ep  1 | Train: 2.1408 | Val: 1.1218 | LR: 0.000997
  Ep  5 | Train: 1.0023 | Val: 0.8535 | LR: 0.000933
  Ep 10 | Train: 0.8651 | Val: 0.7947 | LR: 0.000750
  Ep 15 | Train: 0.7791 | Val: 0.8000 | LR: 0.00